In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

### Importing CrewAI :

In [2]:
from crewai import Agent, Task, Crew, LLM

In [3]:
import os
import google.generativeai as genai
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
from pathlib import Path
# Explicitly point to the .env file next to this notebook
env_path = Path(__file__).parent / ".env" if "__file__" in dir() else Path("e:/AGENTIC AI BOOTCAMP/DEEPLEARNING_COURSES/STAGE_0/3) MULTI AGENT SYSTEMS WITH CREWAI/6) STOCK ANALYSIS AUTOMATION/.env")
load_dotenv(dotenv_path=env_path, override=True)
print("CWD:", os.getcwd())


CWD: e:\AGENTIC AI BOOTCAMP\DEEPLEARNING_COURSES\STAGE_0\3) MULTI AGENT SYSTEMS WITH CREWAI\6) STOCK ANALYSIS AUTOMATION


In [5]:
import os

# Keys are already loaded from .env via load_dotenv()
serper_api_key = os.getenv("SERPER_API_KEY")
print("Serper key loaded:", bool(serper_api_key))


print("GEMINI_API_KEY loaded:", bool(os.getenv("GEMINI_API_KEY")))
print("SERPER_API_KEY loaded:", bool(os.getenv("SERPER_API_KEY")))


Serper key loaded: True
GEMINI_API_KEY loaded: True
SERPER_API_KEY loaded: True


In [6]:
gemini_llm = LLM(
    model="gemini/gemini-3.1-pro-preview",
    temperature=0.0
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


### Importing CrewAI tools : 

In [7]:
from crewai_tools import ScrapeWebsiteTool, SerperDevTool

search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()

### Creating Agents :

In [8]:
data_analyst_agent = Agent(
    role="Data Analyst",
    goal="Monitor and analyze NSE market data for Indian defense stocks "
         "to identify swing and momentum trading opportunities.",
    backstory="Specializing in Indian equity markets (NSE/BSE), this agent "
              "uses technical analysis — RSI, MACD, volume surges, and geopolitical "
              "news signals — to spot swing and momentum setups in defense sector "
              "stocks like HAL, BEL, BDL, Mazagon Dock.",
    verbose=True,
    allow_delegation=True,
    tools=[scrape_tool, search_tool],
    llm=gemini_llm
)



In [9]:
trading_strategy_agent = Agent(
    role="Trading Strategy Developer",
    goal="Develop swing trading and momentum trading strategies for Indian "
         "defense stocks on NSE, leveraging geopolitical catalysts and technical setups.",
    backstory="Expert in Indian equity swing trading (5–30 day holds) and "
              "momentum trading. Identifies breakouts, trend continuations, and "
              "sector rotation signals in defense stocks. Considers Middle East "
              "conflict as a structural tailwind for India's defense exports.",
    verbose=True,
    allow_delegation=True,
    tools=[scrape_tool, search_tool],
    llm=gemini_llm
)


In [10]:
execution_agent = Agent(
    role="Trade Advisor",
    goal="Suggest optimal trade entry and exit strategies for NSE-listed "
         "defense stocks based on approved swing/momentum strategies.",
    backstory="Specializes in NSE trading mechanics — circuit limits, delivery vs "
              "intraday, F&O considerations for defense stocks. Advises on entry "
              "price zones, stop-loss levels, and target prices for swing trades.",
    verbose=True,
    allow_delegation=True,
    tools=[scrape_tool, search_tool],
    llm=gemini_llm
)


In [11]:
risk_management_agent = Agent(
    role="Risk Advisor",
    goal="Evaluate and provide insights on the risks "
         "associated with potential trading activities.",
    backstory="Armed with a deep understanding of risk assessment models "
              "and market dynamics, this agent scrutinizes the potential "
              "risks of proposed trades. It offers a detailed analysis of "
              "risk exposure and suggests safeguards to ensure that "
              "trading activities align with the firm’s risk tolerance.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool],
    llm = gemini_llm
)

### Now time for Creating Tasks to Agents :

In [12]:
# Task for Data Analyst Agent: Analyze Market Data
data_analysis_task = Task(
    description=(
        "Continuously monitor and analyze market data for "
        "the selected stock ({stock_selection}). "
        "Use statistical modeling and machine learning to "
        "identify trends and predict market movements."
    ),
    expected_output=(
        "Insights and alerts about significant market "
        "opportunities or threats for {stock_selection}."
    ),
    agent=data_analyst_agent,
)

In [13]:
# Task for Trading Strategy Agent: Develop Trading Strategies
strategy_development_task = Task(
    description=(
        "Develop and refine trading strategies based on "
        "the insights from the Data Analyst and "
        "user-defined risk tolerance ({risk_tolerance}). "
        "Consider trading preferences ({trading_strategy_preference})."
    ),
    expected_output=(
        "A set of potential trading strategies for {stock_selection} "
        "that align with the user's risk tolerance."
    ),
    agent=trading_strategy_agent,
)


In [14]:
# Task for Trade Advisor Agent: Plan Trade Execution
execution_planning_task = Task(
    description=(
        "Analyze approved trading strategies to determine the "
        "best execution methods for {stock_selection}, "
        "considering current market conditions and optimal pricing."
    ),
    expected_output=(
        "Detailed execution plans suggesting how and when to "
        "execute trades for {stock_selection}."
    ),
    agent=execution_agent,
)

In [15]:
# Task for Risk Advisor Agent: Assess Trading Risks
risk_assessment_task = Task(
    description=(
        "Evaluate the risks associated with the proposed trading "
        "strategies and execution plans for {stock_selection}. "
        "Provide a detailed analysis of potential risks "
        "and suggest mitigation strategies."
    ),
    expected_output=(
        "A comprehensive risk analysis report detailing potential "
        "risks and mitigation recommendations for {stock_selection}."
    ),
    agent=risk_management_agent,
)

### Creating the Crew:

Here w use the `Process` in order to get the information from the agents and process it like the Manager..It is Hierarchial

In [16]:
from crewai import Crew, Process

# Define the crew with agents and tasks
financial_trading_crew = Crew(
    agents=[data_analyst_agent, 
            trading_strategy_agent, 
            execution_agent, 
            risk_management_agent],
    
    tasks=[data_analysis_task, 
           strategy_development_task, 
           execution_planning_task, 
           risk_assessment_task],
    
    manager_llm= LLM(model="gemini/gemini-3.1-pro-preview",
    temperature=0.0),
    process=Process.hierarchical,
    verbose=True
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


### Running the Crew :

In [17]:
## inputs — Indian Defense Stocks: Swing + Momentum Trading
financial_trading_inputs = {
    'stock_selection': 'HAL.NS, BEL.NS, BDL.NS',
    'initial_capital': '500000',                     # ₹5 Lakh INR
    'risk_tolerance': 'Medium',
    'trading_strategy_preference': 'Swing Trading and Momentum Trading',
    'news_impact_consideration': True,
    'market': 'NSE India',
    'sector_focus': 'Defense and Aerospace',
    'geopolitical_context': 'Middle East conflict driving global defense spending surge, India increasing defense budget'
}


In [18]:
## execution time run for sometime for this code below
result = financial_trading_crew.kickoff(inputs=financial_trading_inputs)


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.13.0                                                                                        │
│  Latest version:  1.14.1                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: cc119520-061a-4144-b0a8-e6919bb531ab                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Continuously monitor and analyze market data for the selected stock (HAL.NS, BEL.NS, BDL.NS). Use        │
│  statistical modeling and machine learning to identify trends and predict market movements.                     │
│  ID: d9cd3787-fd19-4bff-86ba-b0cb6684dc28                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Task: Continuously monitor and analyze market data for the selected stock (HAL.NS, BEL.NS, BDL.NS). Use        │
│  statistical modeling and machine learning to identify trends and predict market movements.                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Args: {'task': 'Monitor and analyze market data for HAL.NS, BEL.NS, BDL.NS using statistical modeling and      │
│  machine learning to identify trends and predict market movements. Provide insights and alerts about ...        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────── ✅ Tool Execution Completed (#4) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Output: **To:** Trading Desk / Strategy Team                                                                   │
│  **From:** Risk Advisor                                                                                         │
│  **Subject:** Comprehensive Risk Assessment & Mitigation Plan: Indian Defense Sector Strategies (HAL.NS,        │
│  BEL.NS, BDL.NS)                                                                                                │
│                                                                                                                 │
│  Team,                                                                                                          │
│                                                                                                                 │
│  I have reviewed the proposed swing and momentum trading strategies for Hindustan Aeronautics Limited (HAL),    │
│  Bharat Electronics Limited (BEL), and Bharat Dynamics Limited (BDL). While the structural tailwinds and        │
│  momentum factors present a compelling bullish case, the reliance on exogenous geopolitical catalysts and the   │
│  overextended technical indicators introduce significant vulnerabilities.                                       │
│                                                                                                                 │
│  As your Risk Advisor, my primary concern is ensuring that our exposure remains strictly within the defined     │
│  "Medium Risk" tolerance. Below is a detailed evaluation of the potential risks associated with your execution  │
│  plans, along with mandatory mitigation strategies and safeguards.                                              │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 1. Systemic & Macro-Economic Risks (Sector-Wide)                                                           │
│                                                                                                                 │
│  **Risk: Geopolitical Dependency & The "De-escalation" Threat**                                                 │
│  The current valuation premiums in the defense sector are heavily inflated by geopolitical risk premiums        │
│  (Middle East tensions, US-Iran conflicts). This is an exogenous variable. Any sudden diplomatic resolution,    │
│  ceasefire, or de-escalation will trigger an immediate deflation of this premium, leading to a rapid,           │
│  sector-wide algorithmic sell-off.                                                                              │
│  *   **Mitigation:**                                                                                            │
│      *   **Automated Sentiment Tracking:** Implement strict NLP/News alerts for keywords such as "ceasefire,"   │
│  "diplomatic resolution," or "de-escalation."                                                                   │
│      *   **Emergency Protocol:** If a de-escalation event occurs during market hours, immediately convert CNC   │
│  holdings to MIS and liquidate 50% of the portfolio at market price, while tightening GTT stop-losses on the    │
│  remainder to 1% below the Current Market Price (CMP). 

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Comprehensive Risk Assessment & Mitigation Plan: Indian Defense Sector Strategies (HAL.NS, BEL.NS,           │
│  BDL.NS)**                                                                                                      │
│                                                                                                                 │
│  **To:** Trading Desk / Strategy Team                                                                           │
│  **From:** Risk Advisor                                                                                         │
│  **Subject:** Comprehensive Risk Assessment & Mitigation Plan: Indian Defense Sector Strategies (HAL.NS,        │
│  BEL.NS, BDL.NS)                                                                                                │
│                                                                                                                 │
│  Team,                                                                                                          │
│                                                                                                                 │
│  I have reviewed the proposed swing and momentum trading strategies for Hindustan Aeronautics Limited (HAL),    │
│  Bharat Electronics Limited (BEL), and Bharat Dynamics Limited (BDL). While the structural tailwinds and        │
│  momentum factors present a compelling bullish case, the reliance on exogenous geopolitical catalysts and the   │
│  overextended technical indicators introduce significant vulnerabilities.                                       │
│                                                                                                                 │
│  As your Risk Advisor, my primary concern is ensuring that our exposure remains strictly within the defined     │
│  "Medium Risk" tolerance. Below is a detailed evaluation of the potential risks associated with your execution  │
│  plans, along with mandatory mitigation strategies and safeguards.                                              │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 1. Systemic & Macro-Economic Risks (Sector-Wide)                                                           │
│                                                                                                                 │
│  **Risk: Geopolitical Dependency & The "De-escalation" Threat**                                                 │
│  The current valuation premiums in the defense sector are heavily inflated by geopolitical risk premiums        │
│  (Middle East tensions, US-Iran conflicts). This is an exogenous variable. Any sudden diplomatic resolution,    │
│  ceasefire, or de-escalation will trigger an immediate deflation of this premium, leading to a rapid,           │
│  sector-wide algorithmic sell-off.                                                                              │
│  *   **Mitigation:**                                                                                            │
│      *   **Automated Sentiment Tracking:** Implement st

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Evaluate the risks associated with the proposed trading strategies and execution plans for HAL.NS,       │
│  BEL.NS, BDL.NS. Provide a detailed analysis of potential risks and suggest mitigation strategies.              │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: cc119520-061a-4144-b0a8-e6919bb531ab                                                                       │
│  Final Output: **Comprehensive Risk Assessment & Mitigation Plan: Indian Defense Sector Strategies (HAL.NS,     │
│  BEL.NS, BDL.NS)**                                                                                              │
│                                                                                                                 │
│  **To:** Trading Desk / Strategy Team                                                                           │
│  **From:** Risk Advisor                                                                                         │
│  **Subject:** Comprehensive Risk Assessment & Mitigation Plan: Indian Defense Sector Strategies (HAL.NS,        │
│  BEL.NS, BDL.NS)                                                                                                │
│                                                                                                                 │
│  Team,                                                                                                          │
│                                                                                                                 │
│  I have reviewed the proposed swing and momentum trading strategies for Hindustan Aeronautics Limited (HAL),    │
│  Bharat Electronics Limited (BEL), and Bharat Dynamics Limited (BDL). While the structural tailwinds and        │
│  momentum factors present a compelling bullish case, the reliance on exogenous geopolitical catalysts and the   │
│  overextended technical indicators introduce significant vulnerabilities.                                       │
│                                                                                                                 │
│  As your Risk Advisor, my primary concern is ensuring that our exposure remains strictly within the defined     │
│  "Medium Risk" tolerance. Below is a detailed evaluation of the potential risks associated with your execution  │
│  plans, along with mandatory mitigation strategies and safeguards.                                              │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 1. Systemic & Macro-Economic Risks (Sector-Wide)                                                           │
│                                                                                                                 │
│  **Risk: Geopolitical Dependency & The "De-escalation" Threat**                                                 │
│  The current valuation premiums in the defense sector are heavily inflated by geopolitical risk premiums        │
│  (Middle East tensions, US-Iran conflicts). This is an exogenous variable. Any sudden diplomatic resolution,    │
│  ceasefire, or de-escalation will trigger an immediate deflation of this premium, leading to a rapid,           │
│  sector-wide algorithmic sell-off.                                                                              │
│  *   **Mitigation:**                                                                                            │
│      *   **Automated Sentiment Tracking:** Implement s



┌───────────────────────── Tracing Preference Saved ──────────────────────────┐
│                                                                             │
│  Info: Tracing has been disabled.                                           │
│                                                                             │
│  Your preference has been saved. Future Crew/Flow executions will not       │
│  collect traces.                                                            │
│                                                                             │
│  To enable tracing later, do any one of these:                              │
│  • Set tracing=True in your Crew/Flow code                                  │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file              │
│  • Run: crewai traces enable                                                │
│                                                                             │
└─────────────────────────────────────

In [19]:
## Markdowning the Result 
from IPython.display import Markdown
Markdown(result.raw)

**Comprehensive Risk Assessment & Mitigation Plan: Indian Defense Sector Strategies (HAL.NS, BEL.NS, BDL.NS)**

**To:** Trading Desk / Strategy Team
**From:** Risk Advisor
**Subject:** Comprehensive Risk Assessment & Mitigation Plan: Indian Defense Sector Strategies (HAL.NS, BEL.NS, BDL.NS)

Team,

I have reviewed the proposed swing and momentum trading strategies for Hindustan Aeronautics Limited (HAL), Bharat Electronics Limited (BEL), and Bharat Dynamics Limited (BDL). While the structural tailwinds and momentum factors present a compelling bullish case, the reliance on exogenous geopolitical catalysts and the overextended technical indicators introduce significant vulnerabilities. 

As your Risk Advisor, my primary concern is ensuring that our exposure remains strictly within the defined "Medium Risk" tolerance. Below is a detailed evaluation of the potential risks associated with your execution plans, along with mandatory mitigation strategies and safeguards.

---

### 1. Systemic & Macro-Economic Risks (Sector-Wide)

**Risk: Geopolitical Dependency & The "De-escalation" Threat**
The current valuation premiums in the defense sector are heavily inflated by geopolitical risk premiums (Middle East tensions, US-Iran conflicts). This is an exogenous variable. Any sudden diplomatic resolution, ceasefire, or de-escalation will trigger an immediate deflation of this premium, leading to a rapid, sector-wide algorithmic sell-off.
*   **Mitigation:** 
    *   **Automated Sentiment Tracking:** Implement strict NLP/News alerts for keywords such as "ceasefire," "diplomatic resolution," or "de-escalation." 
    *   **Emergency Protocol:** If a de-escalation event occurs during market hours, immediately convert CNC holdings to MIS and liquidate 50% of the portfolio at market price, while tightening GTT stop-losses on the remainder to 1% below the Current Market Price (CMP).

**Risk: Sector Concentration & Correlation**
Allocating capital across HAL, BEL, and BDL creates highly correlated exposure. A negative sector-specific catalyst will impact all three positions simultaneously.
*   **Mitigation:** Strictly adhere to the proposed portfolio cap. The combined allocation to the defense sector **must not exceed 15-20%** of the total swing trading portfolio. 

**Risk: Implied Volatility (IV) Crush in F&O**
Geopolitical tensions have artificially inflated Implied Volatility. 
*   **Mitigation:** The strategy’s mandate to use **Cash Delivery (CNC)** instead of options is fully endorsed from a risk perspective. Buying options in this environment guarantees exposure to severe IV crush upon any stabilization of news flow. If leverage via Stock Futures is utilized, it must be strictly monitored to prevent margin calls during intraday volatility spikes.

---

### 2. Stock-Specific Risk Analysis & Execution Safeguards

#### A. Hindustan Aeronautics Limited (HAL.NS) - *Momentum Continuation*
*   **Identified Risks:** HAL is technically overextended. With a Stochastic RSI > 80, the probability of a mean-reversion pullback is high. The primary risk in the "Buy-the-Dip" strategy is mistaking a structural trend reversal for a temporary pullback (catching a falling knife). Furthermore, dynamic price bands (lack of hard circuit limits) expose us to severe intraday volatility.
*   **Mitigation & Safeguards:**
    *   **Entry Discipline:** Do not execute market orders. The GTT Buy order at a 3-5% pullback to the 10-day or 20-day EMA is mandatory. 
    *   **Hard Stop-Loss:** The 4% GTT Stop-Loss must be non-negotiable. If the price slices through the 20-day EMA on high volume, the momentum thesis is invalidated.
    *   **Trailing Mechanism:** Trailing the stop-loss to breakeven at a 5% unrealized gain is an excellent risk-reduction tactic and must be strictly enforced.

#### B. Bharat Electronics Limited (BEL.NS) - *Range-Bound Swing*
*   **Identified Risks:** BEL presents the highest fundamental and technical divergence risk. It is trading at a premium valuation, yet exhibits a weak trend (ADX ~16.22) and an extremely overbought Stochastic RSI (>90). This makes it highly susceptible to sudden profit-booking or prolonged sideways chop, which ties up capital (opportunity cost).
*   **Mitigation & Safeguards:**
    *   **Tight Leash:** The proposed 3-4% stop-loss is appropriate given the weak ADX. Do not give this trade room to breathe if it breaks below the 20-day SMA.
    *   **Aggressive De-risking:** The OCO order to scale out 50% of the position at a 5% gain is crucial. Securing partial profits early protects against the high probability of mean reversion.
    *   **Indicator Monitoring:** If daily MACD negative divergence appears, exit the remaining position manually at 3:15 PM. Do not wait for the trailing stop to hit.

#### C. Bharat Dynamics Limited (BDL.NS) - *Volatility Breakout*
*   **Identified Risks:** BDL is in a consolidation phase with mixed signals. The primary risk here is a **False Breakout (Bull Trap)**—where the price breaches resistance intraday but fails to hold, trapping breakout buyers before reversing sharply.
*   **Mitigation & Safeguards:**
    *   **Volume Verification:** The automated Stop-Limit Buy order is risky on its own. The 3:15 PM volume check (must be >1.5x the 20-day average) is a mandatory safeguard. If volume is weak, cancel the trade.
    *   **The "Look Above and Fail" Rule:** If BDL triggers the entry but closes back inside the consolidation range the following day, manually square off the position immediately. Do not wait for the 5% stop-loss to trigger, as the technical setup has already failed.

---

### 3. Operational & Execution Risks

**Risk: Gap-Downs Bypassing GTT Orders**
GTT (Good Till Triggered) stop-loss orders are executed as limit or market orders *after* the trigger price is hit. In the event of a massive overnight gap-down (due to global news or poor earnings), the opening price may be significantly below your stop-loss, resulting in a much larger loss than the planned 3-5%.
*   **Mitigation (Earnings Roulette):** Never hold a full-sized position overnight into an earnings print. Manually liquidate 50% of the position at 3:15 PM the day prior to any scheduled corporate earnings announcement for HAL, BEL, or BDL. 

### Final Verdict
The proposed trading plan is well-structured and correctly identifies the technical realities of the market. However, its success hinges entirely on strict adherence to the execution mechanics. The overbought nature of these equities means the margin for error is slim. 

**Approval Status:** Approved for execution, provided the strict position sizing (max 20% sector exposure), volume-confirmation rules, and geopolitical news alerts are actively managed by the desk.